# 02 — Exploratory Data Analysis

## Objetivo

Analizar el comportamiento de compra de los usuarios de Instacart con foco en dos preguntas de negocio:

1. ¿Qué productos y categorías impulsan la recompra?
2. ¿Qué combinaciones de productos presentan oportunidades de cross-sell?

El análisis se concentra en:

- KPIs generales del dataset.
- Comportamiento temporal de los pedidos.
- Productos con mayor volumen de compra.
- Productos con mayor tasa de recompra.
- Recompra por departamento y aisle.
- Asociaciones de productos mediante métricas de Market Basket Analysis.

## Preparación y alcance

Ejecutá todas las celdas en orden con **Restart Kernel and Run All**. Este notebook necesita `pandas`, `matplotlib` y `duckdb` en el mismo entorno de Python que el kernel de Jupyter. Si falta alguna biblioteca, ejecutá en una celda `%pip install pandas matplotlib duckdb` y reiniciá el kernel.

Los CSV originales deben estar en `data/raw`. Las cinco tablas de `data/processed` se generan en la sección de preparación de `01_data_profiling.ipynb`; los scripts SQL actuales solo muestran resultados y no exportan estas tablas.

**Universos:** usuarios, pedidos e intervalos generales y las distribuciones temporales se calculan sobre `orders` (`prior`, `train` y `test`). Las canastas, los productos, la recompra y las asociaciones usan exclusivamente `prior`. Una línea es un producto dentro de un pedido, no unidades ni valor monetario. `reordered = 1` indica que ese usuario ya compró ese producto anteriormente.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import duckdb

# Permite ejecutar desde la raíz del repositorio o desde notebooks/.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'data' / 'raw' / 'orders.csv').is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Abrí Jupyter desde este repositorio y colocá los CSV en data/raw.'
    )
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
required = [
    PROCESSED_DIR / 'dim_products.csv',
    PROCESSED_DIR / 'order_basket_summary.csv',
    PROCESSED_DIR / 'product_reorder_metrics.csv',
    PROCESSED_DIR / 'aisle_reorder_metrics.csv',
    PROCESSED_DIR / 'department_reorder_metrics.csv',
    RAW_DIR / 'orders.csv',
    RAW_DIR / 'order_products__prior.csv',
    RAW_DIR / 'products.csv',
]
missing_files = [str(p) for p in required if not p.is_file()]
if missing_files:
    raise FileNotFoundError(
        'Faltan archivos: ' + ', '.join(missing_files)
        + '. Descargá los CSV y ejecutá la preparación de tablas de 01_data_profiling.ipynb.'
    )

dim_products = pd.read_csv(PROCESSED_DIR / 'dim_products.csv')
order_basket_summary = pd.read_csv(PROCESSED_DIR / 'order_basket_summary.csv')
product_reorder_metrics = pd.read_csv(PROCESSED_DIR / 'product_reorder_metrics.csv')
aisle_reorder_metrics = pd.read_csv(PROCESSED_DIR / 'aisle_reorder_metrics.csv')
department_reorder_metrics = pd.read_csv(PROCESSED_DIR / 'department_reorder_metrics.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')

# Comprobar que las tablas agregadas representan el mismo histórico.
assert dim_products['product_id'].is_unique
assert order_basket_summary['order_id'].is_unique
assert product_reorder_metrics['product_id'].is_unique
assert len(order_basket_summary) == (orders['eval_set'] == 'prior').sum()
assert order_basket_summary['basket_size'].sum() == product_reorder_metrics['total_purchases'].sum()
assert order_basket_summary['reordered_items'].sum() == product_reorder_metrics['reordered_purchases'].sum()
for category_metrics in [aisle_reorder_metrics, department_reorder_metrics]:
    assert category_metrics['total_purchases'].sum() == product_reorder_metrics['total_purchases'].sum()
    assert category_metrics['reordered_purchases'].sum() == product_reorder_metrics['reordered_purchases'].sum()

print('Datasets cargados y totales reconciliados correctamente.')


In [ ]:
total_users = orders['user_id'].nunique()

total_orders = orders['order_id'].nunique()

total_products = dim_products['product_id'].nunique()

total_items = product_reorder_metrics['total_purchases'].sum()

total_reordered = (
    product_reorder_metrics['reordered_purchases'].sum()
)

global_reorder_rate = (
    total_reordered / total_items
)

avg_basket_size = (
    order_basket_summary['basket_size'].mean()
)

avg_days_between_orders = (
    orders['days_since_prior_order'].mean()
)

print("USUARIOS:", total_users)
print("PEDIDOS:", total_orders)
print("PRODUCTOS:", total_products)
print("ÍTEMS COMPRADOS:", total_items)

print(
    "TASA GLOBAL DE RECOMPRA:",
    round(global_reorder_rate * 100, 2),
    "%"
)

print(
    "TAMAÑO PROMEDIO DEL CARRITO:",
    round(avg_basket_size, 2)
)

print(
    "DÍAS PROMEDIO ENTRE PEDIDOS:",
    round(avg_days_between_orders, 2)
)

## KPIs generales — ¿Cuál es la escala del negocio observado?

Se usa `nunique()` para contar identificadores distintos, `sum()` para las compras y recompras agregadas, y `mean()` para las canastas y los intervalos. Los promedios de pandas omiten los nulos; el primer pedido de cada usuario no tiene intervalo anterior y no se imputa.

- **206.209 usuarios**, **3.421.083 pedidos** (`prior`, `train`, `test`) y **49.688 productos** en el catálogo.
- **32.434.489 líneas de producto** en **3.214.874 pedidos históricos `prior`**.
- **58,97 % de recompra**: suma de líneas recompradas / suma de líneas compradas. Es una tasa ponderada por líneas; no es el porcentaje de usuarios que vuelve ni el promedio de tasas individuales de productos.
- **10,09 productos por canasta histórica**, promediando cada pedido por igual.
- **11,11 días de intervalo registrado** en `orders`, excluyendo nulos. La variable observada está limitada a valores de 0 a 30; este promedio no permite reconstruir todos los intervalos reales ni proyectar una frecuencia exacta.

La recompra ocupa una parte sustancial de las compras observadas y justifica estudiar productos recurrentes. Estos KPIs combinan universos distintos de forma explícita: no se divide el número de líneas `prior` por el total de pedidos que también contiene `train` y `test`.


## Patrones temporales — ¿Cuándo se registran más pedidos?

Se agrupa `orders` por `order_dow` y `order_hour_of_day`; `.size()` cuenta pedidos en cada grupo. El gráfico de barras compara días codificados y el de líneas muestra el perfil horario. Se incluyen todos los tipos de pedido porque estas columnas existen en `orders`.


In [ ]:
orders_by_day = (
    orders
    .groupby('order_dow')
    .size()
    .reset_index(name='total_orders')
)

orders_by_day

In [ ]:
plt.figure(figsize=(8, 4))

plt.bar(
    orders_by_day['order_dow'],
    orders_by_day['total_orders']
)

plt.title('Pedidos por día de la semana')
plt.xlabel('Día codificado (0–6)')
plt.ylabel('Cantidad de pedidos')

plt.tight_layout()
plt.show()

In [ ]:
orders_by_hour = (
    orders
    .groupby('order_hour_of_day')
    .size()
    .reset_index(name='total_orders')
)

orders_by_hour

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(
    orders_by_hour['order_hour_of_day'],
    orders_by_hour['total_orders'],
    marker='o'
)

plt.title('Pedidos por hora del día')
plt.xlabel('Hora')
plt.ylabel('Cantidad de pedidos')

plt.xticks(range(24))

plt.tight_layout()
plt.show()

### Interpretación temporal

Los códigos **0 y 1** reúnen **600.905** y **587.478 pedidos**, respectivamente: juntos representan **34,74 %** del total. El menor volumen se registra en el código 4 (**426.339**). No se asignan nombres de días calendario a estos códigos sin una equivalencia documentada.

El máximo horario está a las **10 h, con 288.418 pedidos**; entre las **9 y las 17 h**, inclusive, se concentra **71,61 %** de los pedidos. Esto sugiere franjas para evaluar capacidad operativa o exposición de recomendaciones. Se trata de una distribución agregada: no demuestra tendencias por fecha, estacionalidad ni horarios óptimos para cada usuario.


## Volumen — ¿Qué productos se compran más?

Se ordena `product_reorder_metrics` por `total_purchases` de mayor a menor y se seleccionan los primeros 15. La tabla compara compras, recompras y tasa; las barras horizontales facilitan leer los nombres. Se invierte el orden al dibujar para que el líder aparezca arriba.


In [ ]:
top_products = (
    product_reorder_metrics
    .sort_values(
        'total_purchases',
        ascending=False
    )
    .head(15)
)

top_products[[
    'product_name',
    'total_purchases',
    'reordered_purchases',
    'reorder_rate'
]]

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    top_products['product_name'][::-1],
    top_products['total_purchases'][::-1]
)

plt.title('Top 15 productos más comprados')
plt.xlabel('Cantidad de compras')
plt.ylabel('Producto')

plt.tight_layout()
plt.show()

### Interpretación del volumen

**Banana** lidera con **472.565 compras**, seguida por **Bag of Organic Bananas** (**379.450**) y **Organic Strawberries** (**264.683**). También aparecen espinaca, palta, cítricos y leche: los productos frescos ocupan buena parte del ranking.

La tasa de Banana es **84,35 %** y la de Bag of Organic Bananas **83,26 %**. Su combinación de alcance y recurrencia las vuelve candidatas para estudiar disponibilidad y accesos rápidos a compras habituales. Las cantidades representan líneas/pedidos con ese producto, no kilos, unidades físicas, ingresos ni márgenes.


## Recurrencia — ¿Qué productos tienen mayor tasa de recompra con volumen suficiente?

Primero se filtran los productos con **al menos 10.000 compras históricas**. Después se ordenan por `reorder_rate` y se muestran los primeros 15. El umbral reduce la presencia de tasas altas basadas en pocos registros, pero no equivale a una prueba de significancia estadística. La tasa se multiplica por 100 solo para expresarla como porcentaje en el gráfico.


In [ ]:
high_volume_products = (
    product_reorder_metrics[
        product_reorder_metrics['total_purchases'] >= 10000
    ]
    .sort_values(
        'reorder_rate',
        ascending=False
    )
)

high_volume_products[[
    'product_name',
    'total_purchases',
    'reordered_purchases',
    'reorder_rate'
]].head(15)

In [ ]:
top_reorder_products = high_volume_products.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_reorder_products['product_name'][::-1],
    top_reorder_products['reorder_rate'][::-1] * 100
)

plt.title('Productos con mayor tasa de recompra')
plt.xlabel('Tasa de recompra (%)')
plt.ylabel('Producto')

plt.tight_layout()
plt.show()

### Interpretación de la recurrencia

Con el piso de 10.000 compras, **Milk, Organic, Vitamin D** tiene la mayor tasa (**85,43 %**, 20.198 compras), seguida de **Organic Reduced Fat Milk** (**85,07 %**, 35.663). Banana ocupa el tercer lugar (**84,35 %**) con un volumen mucho mayor.

El ranking destaca leche, yogur y agua además de bananas. Puede orientar una experiencia de reposición de productos habituales. Una tasa alta describe recurrencia histórica; no demuestra que una campaña de descuento aumente la compra ni identifica por sí sola al producto de mayor impacto comercial.


## Departamentos — ¿Qué categorías concentran la recompra?

Se ordenan los departamentos por su tasa agregada y se comparan las 21 categorías, incluyendo `missing`. Para cada departamento, la tasa es recompras / compras de todas sus líneas; no se promedian por igual las tasas de cada producto. La tabla de volumen permite contextualizar el ranking de tasas.


In [ ]:
department_analysis = (
    department_reorder_metrics
    .sort_values(
        'reorder_rate',
        ascending=False
    )
)

department_analysis

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    department_analysis['department'][::-1],
    department_analysis['reorder_rate'][::-1] * 100
)

plt.title('Tasa de recompra por departamento')
plt.xlabel('Tasa de recompra (%)')
plt.ylabel('Departamento')

plt.tight_layout()
plt.show()

In [ ]:
department_analysis[[
    'department',
    'total_purchases',
    'reordered_purchases',
    'reorder_rate'
]]

### Interpretación por departamento

**dairy eggs** lidera la tasa (**67,00 %**), seguido por **beverages** (**65,35 %**) y **produce** (**64,99 %**). Sin embargo, **produce** concentra el mayor volumen: **9.479.291 compras** y **6.160.710 recompras**, frente a **5.414.016 compras** de dairy eggs.

**personal care** tiene la menor tasa (**32,11 %**). Una tasa baja puede reflejar distintas necesidades de reposición y composición del surtido; no demuestra insatisfacción. La categoría `missing` se conserva (**39,58 %**) para no ocultar productos sin clasificación y debe tratarse como una limitación del catálogo.


## Pasillos — ¿Dónde está la recurrencia dentro de las categorías?

Los `aisles` representan una clasificación más específica que los departamentos. Se exige **al menos 10.000 compras** por pasillo y se muestran los 15 con mayor tasa, tanto en tabla como en barras. Este nivel ayuda a pasar de una categoría amplia a una selección comercial más concreta.


In [ ]:
aisle_analysis = (
    aisle_reorder_metrics[
        aisle_reorder_metrics['total_purchases'] >= 10000
    ]
    .sort_values(
        'reorder_rate',
        ascending=False
    )
)

aisle_analysis.head(15)

In [ ]:
top_aisles = aisle_analysis.head(15)

plt.figure(figsize=(10, 6))

plt.barh(
    top_aisles['aisle'][::-1],
    top_aisles['reorder_rate'][::-1] * 100
)

plt.title('Aisles con mayor tasa de recompra')
plt.xlabel('Tasa de recompra (%)')
plt.ylabel('Aisle')

plt.tight_layout()
plt.show()

### Interpretación por pasillo

**milk** alcanza **78,14 %** de recompra; **water seltzer sparkling water**, **72,96 %**; **fresh fruits**, **71,81 %**; y **eggs**, **70,54 %**. Fresh fruits además registra **3.642.188 compras**, de modo que combina una tasa elevada con gran volumen.

Este desglose concreta el hallazgo de los departamentos: permite explorar acciones sobre leche, agua, frutas y huevos en lugar de tratar a todos sus productos de la misma manera.


## Market Basket Analysis — ¿Qué productos se compran juntos?

DuckDB procesa el CSV histórico mediante SQL y devuelve un DataFrame de pandas. Se registran vistas sobre los archivos, sin cargar primero las 32 millones de líneas en un DataFrame. Las CTEs construyen el análisis por etapas:

1. `product_orders` cuenta pedidos distintos por producto.
2. `top_products` selecciona los **500 productos con presencia en más pedidos** para limitar el costo del autojoin.
3. `basket` conserva sus apariciones en los pedidos.
4. `pairs` une la canasta consigo misma por `order_id`. `a.product_id < b.product_id` evita pares de un producto consigo mismo y las dos orientaciones del mismo par; se cuentan pedidos distintos.
5. `total_orders` calcula todos los pedidos `prior`, incluso aquellos sin productos del top 500, para mantener el denominador correcto.
6. Los joins incorporan nombres y frecuencias individuales; se conservan pares presentes en **al menos 100 pedidos** y se ordenan por frecuencia conjunta.

Para un par A–B, con N pedidos históricos:

| Métrica | Cálculo | Interpretación |
| --- | --- | --- |
| Soporte | pedidos con A y B / N | Alcance del par en todos los pedidos históricos. |
| Confianza A → B | pedidos con A y B / pedidos con A | Proporción de pedidos con A que también contienen B. |
| Confianza B → A | pedidos con A y B / pedidos con B | La dirección inversa puede tener otra proporción. |
| Lift | confianza A → B / (pedidos con B / N) | Frecuencia relativa frente a la esperada bajo independencia. Mayor que 1 indica asociación positiva; cercano a 1, ausencia de asociación; menor que 1, asociación negativa. |

El `lift` es simétrico aunque la confianza no lo sea. Las direcciones A/B dependen del orden de los identificadores y no representan una secuencia de compra ni una relación causal. Se comprueban pares únicos y que soporte y confianza estén entre 0 y 1.


In [ ]:
con = duckdb.connect(database=':memory:')
# Registrar rutas sin interpolarlas en la consulta SQL.
con.read_csv(str(RAW_DIR / 'order_products__prior.csv')).create_view('prior_items')
con.read_csv(str(RAW_DIR / 'products.csv')).create_view('products_catalog')

cross_sell = con.execute("""
WITH product_orders AS (

    SELECT
        product_id,
        COUNT(DISTINCT order_id) AS product_orders

    FROM prior_items

    GROUP BY product_id
),

top_products AS (

    SELECT
        product_id,
        product_orders

    FROM product_orders

    ORDER BY product_orders DESC

    LIMIT 500
),

basket AS (

    SELECT
        op.order_id,
        op.product_id

    FROM prior_items op

    INNER JOIN top_products tp
        ON op.product_id = tp.product_id
),

pairs AS (

    SELECT
        a.product_id AS product_id_1,
        b.product_id AS product_id_2,
        COUNT(DISTINCT a.order_id) AS orders_together

    FROM basket a

    INNER JOIN basket b
        ON a.order_id = b.order_id
        AND a.product_id < b.product_id

    GROUP BY
        a.product_id,
        b.product_id
),

total_orders AS (

    SELECT
        COUNT(DISTINCT order_id) AS total_orders

    FROM prior_items
)

SELECT

    p.product_id_1,
    prod1.product_name AS product_1,

    p.product_id_2,
    prod2.product_name AS product_2,

    p.orders_together,

    tp1.product_orders AS orders_product_1,
    tp2.product_orders AS orders_product_2,

    p.orders_together * 1.0
        / t.total_orders AS support,

    p.orders_together * 1.0
        / tp1.product_orders AS confidence_1_to_2,

    p.orders_together * 1.0
        / tp2.product_orders AS confidence_2_to_1,

    (
        p.orders_together * 1.0
        / tp1.product_orders
    )
    /
    (
        tp2.product_orders * 1.0
        / t.total_orders
    ) AS lift

FROM pairs p

INNER JOIN top_products tp1
    ON p.product_id_1 = tp1.product_id

INNER JOIN top_products tp2
    ON p.product_id_2 = tp2.product_id

INNER JOIN products_catalog prod1
    ON p.product_id_1 = prod1.product_id

INNER JOIN products_catalog prod2
    ON p.product_id_2 = prod2.product_id

CROSS JOIN total_orders t

WHERE p.orders_together >= 100

ORDER BY p.orders_together DESC

""").df()

con.close()
assert cross_sell[['product_id_1', 'product_id_2']].duplicated().sum() == 0
assert (cross_sell['product_id_1'] < cross_sell['product_id_2']).all()
assert cross_sell['support'].between(0, 1).all()
assert cross_sell['confidence_1_to_2'].between(0, 1).all()
assert cross_sell['confidence_2_to_1'].between(0, 1).all()
cross_sell.head()

In [ ]:
cross_sell.to_csv(PROCESSED_DIR / 'cross_sell_metrics.csv', index=False)
print('cross_sell_metrics.csv generado correctamente')


### Exportación

Se genera `data/processed/cross_sell_metrics.csv` con los identificadores, nombres, pedidos conjuntos, frecuencias individuales, soporte, las dos confianzas y lift. En esta ejecución contiene **81.646 pares** con al menos 100 pedidos compartidos. Es una tabla de pares agregados para análisis/BI, no una tabla de pedidos individuales. Los datos originales no se modifican.


### Ranking por alcance

Se muestran los 20 pares con más pedidos compartidos. Este ranking responde qué combinaciones alcanzan a más canastas; no basta por sí solo para afirmar que los productos tienen una afinidad especial, porque la popularidad individual también aumenta la coincidencia.


In [ ]:
cross_sell[
    [
        'product_1',
        'product_2',
        'orders_together',
        'support',
        'confidence_1_to_2',
        'confidence_2_to_1',
        'lift'
    ]
].sort_values(
    'orders_together',
    ascending=False
).head(20)

### Ranking por asociación relativa

Se exige un piso de **1.000 pedidos conjuntos**, se ordena por `lift` y se muestran los primeros 20 pares y un gráfico de los primeros 10. Se combinan así una asociación relativa alta y un mínimo de alcance. Los nombres del gráfico se distribuyen en varias líneas para conservar su legibilidad.


In [ ]:
strong_pairs = (
    cross_sell[
        cross_sell['orders_together'] >= 1000
    ]
    .sort_values(
        'lift',
        ascending=False
    )
)

strong_pairs[[
    'product_1',
    'product_2',
    'orders_together',
    'confidence_1_to_2',
    'confidence_2_to_1',
    'lift'
]].head(20)

In [ ]:
import textwrap

top_pairs = strong_pairs.head(10).copy()

top_pairs['pair'] = (
    top_pairs['product_1']
    + ' + '
    + top_pairs['product_2']
)

top_pairs['pair_label'] = top_pairs['pair'].map(
    lambda label: textwrap.fill(label, width=55)
)

plt.figure(figsize=(12, 10))

plt.barh(
    top_pairs['pair_label'][::-1],
    top_pairs['lift'][::-1]
)

plt.title('Principales oportunidades de cross-sell')
plt.xlabel('Lift')
plt.ylabel('Combinación de productos')

plt.tight_layout()
plt.show()

### Interpretación de las asociaciones

El par más frecuente es **Bag of Organic Bananas + Organic Hass Avocado**, presente en **62.341 pedidos**: soporte **1,94 %**, confianza bananas → palta **16,43 %**, confianza palta → bananas **29,19 %** y lift **2,47**. La dirección desde la palta ofrece mayor confianza, aunque ambos productos forman el mismo par.

Entre los **9.726 pares** con al menos 1.000 pedidos conjuntos, el mayor lift corresponde a **Icelandic Style Skyr Blueberry Non-fat Yogurt + Non Fat Raspberry Yogurt**: **7.224 pedidos**, soporte **0,22 %**, confianzas **37,51 %** y **44,11 %**, y lift **73,64**. Su coincidencia es aproximadamente 73,64 veces la esperada bajo independencia; **no** significa un crecimiento de ventas de ese tamaño. La baja frecuencia individual de los productos hace posible un lift muy alto con un alcance total reducido.

Los primeros puestos por lift son principalmente combinaciones de yogures. Son candidatos para probar recomendaciones, pero no todos los pares frecuentes tienen asociación positiva: **Banana + Organic Hass Avocado** (la banana individual, no la bolsa orgánica) tiene lift cercano a **0,99** pese a aparecer en **31.222 pedidos**. Esto muestra por qué se necesitan soporte, confianza y lift en conjunto.

### Conclusiones y próximos pasos

1. Evaluar disponibilidad y accesos a reposición para productos que combinan volumen y recurrencia, como bananas, leche y frutas.
2. Comparar alternativas de recomendación: pares de amplio alcance (bananas orgánicas/palta) frente a pares de afinidad elevada (yogures).
3. Usar ambas direcciones de confianza para decidir qué recomendar cuando el producto disparador ya está en la canasta.
4. Llevar los KPIs y rankings a Power BI manteniendo visibles sus denominadores y umbrales.
5. Validar las propuestas con un experimento o período de evaluación antes de afirmar impacto: medir aceptación, conversión e incremento de compras; incorporar precios/márgenes si se busca rentabilidad.
